# Clickbait GBDT (LightGBM) - Colab

Drive-mount workflow for VSCode + Colab kernel. Run cells top to bottom.


In [1]:
import os
import sys
import subprocess
import inspect
import inspect
from pathlib import Path

# Optional: set your exact project path in MyDrive to skip auto search
PROJECT_DIR_HINT = '/content/drive/MyDrive/sw_project/clickbait_gbdt_tfidf'

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

def looks_like_project(d: Path) -> bool:
    return (d / 'requirements-colab.txt').exists() and (d / 'src').exists() and (d / 'data' / 'splits').exists()

candidates = []
if PROJECT_DIR_HINT:
    candidates.append(Path(PROJECT_DIR_HINT))
base = Path('/content/drive/MyDrive')
candidates += [base / 'sw_project' / 'clickbait_gbdt_tfidf', base / 'clickbait_gbdt_tfidf']

project_dir = next((d for d in candidates if d.exists() and looks_like_project(d)), None)

if project_dir is None:
    for req in base.rglob('requirements-colab.txt'):
        cand = req.parent
        if looks_like_project(cand):
            project_dir = cand
            break

if project_dir is None:
    raise FileNotFoundError(
        'Project not found in Google Drive. Put clickbait_gbdt_tfidf under MyDrive and set PROJECT_DIR_HINT if needed.'
    )

os.chdir(project_dir)
print('Working directory:', Path.cwd())
print('Python:', sys.version.split()[0])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
print('Dependencies installed')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/sw_project/clickbait_gbdt_tfidf
Python: 3.12.13
Dependencies installed


In [2]:
from pathlib import Path
import pandas as pd

TRAIN_DATA = 'data/splits/train.csv'
VALID_DATA = 'data/splits/valid.csv'
TEST_DATA = 'data/splits/test.csv'
TEXT_COL = 'text'
LABEL_COL = 'label'

for p in [TRAIN_DATA, VALID_DATA, TEST_DATA]:
    print(p, Path(p).exists())

train_cols = pd.read_csv(TRAIN_DATA, nrows=1).columns.tolist()
print('train cols:', train_cols)


data/splits/train.csv True
data/splits/valid.csv True
data/splits/test.csv True
train cols: ['text', 'label', 'label_name', 'folder', 'news_id', 'raw_clickbait_class', 'source_rel_path']


In [3]:
import inspect
from pathlib import Path
import json
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

from src.data_utils import load_training_data
from src.model import build_model


def evaluate_split(name, y_true, y_pred):
    metrics = {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted')),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }
    print(f"\n[{name}] Macro F1: {metrics['macro_f1']:.4f}")
    print(f"[{name}] Weighted F1: {metrics['weighted_f1']:.4f}")
    print(f"[{name}] Accuracy: {metrics['accuracy']:.4f}")
    print(f"[{name}] Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))
    return metrics


SVD_COMPONENTS = 96
N_ESTIMATORS = 120
LEARNING_RATE = 0.05
MAX_DEPTH = 5
NUM_LEAVES = 63
RANDOM_STATE = 42
WORD_MAX_FEATURES = 50000
CHAR_MAX_FEATURES = 0
N_JOBS = 2
USE_CHAR_FEATURES = False
TRAIN_SAMPLE_SIZE = 120000
VALID_SAMPLE_SIZE = 30000
TEST_SAMPLE_SIZE = 30000

MODEL_OUT = Path('models/gbdt_lightgbm_tfidf_svd.joblib')
METRICS_OUT = Path('models/gbdt_metrics.json')

x_train, y_train = load_training_data(TRAIN_DATA, TEXT_COL, LABEL_COL)
x_valid, y_valid = load_training_data(VALID_DATA, TEXT_COL, LABEL_COL)
x_test, y_test = load_training_data(TEST_DATA, TEXT_COL, LABEL_COL)

def stratified_cap(x, y, cap, random_state):
    if cap is None or len(x) <= cap:
        return x.reset_index(drop=True), y.reset_index(drop=True)
    x_small, _, y_small, _ = train_test_split(
        x,
        y,
        train_size=cap,
        random_state=random_state,
        stratify=y,
    )
    return x_small.reset_index(drop=True), y_small.reset_index(drop=True)

x_train, y_train = stratified_cap(x_train, y_train, TRAIN_SAMPLE_SIZE, RANDOM_STATE)
x_valid, y_valid = stratified_cap(x_valid, y_valid, VALID_SAMPLE_SIZE, RANDOM_STATE)
x_test, y_test = stratified_cap(x_test, y_test, TEST_SAMPLE_SIZE, RANDOM_STATE)

print('Model type: lightgbm (GBDT)')
print(f'Train samples: {len(x_train):,}')
print(f'Valid samples: {len(x_valid):,}')
print(f'Test samples: {len(x_test):,}')
print(f'Settings: svd={SVD_COMPONENTS}, estimators={N_ESTIMATORS}, max_depth={MAX_DEPTH}, word_max={WORD_MAX_FEATURES}, char_max={CHAR_MAX_FEATURES}, use_char={USE_CHAR_FEATURES}, n_jobs={N_JOBS}')
print(f'Sampled sizes: train={len(x_train):,}, valid={len(x_valid):,}, test={len(x_test):,}')

build_kwargs = {
    'model_type': 'lightgbm',
    'random_state': RANDOM_STATE,
    'svd_components': SVD_COMPONENTS,
    'n_estimators': N_ESTIMATORS,
    'learning_rate': LEARNING_RATE,
    'max_depth': MAX_DEPTH,
    'num_leaves': NUM_LEAVES,
}
sig = inspect.signature(build_model)
if 'word_max_features' in sig.parameters:
    build_kwargs['word_max_features'] = WORD_MAX_FEATURES
if 'char_max_features' in sig.parameters:
    build_kwargs['char_max_features'] = CHAR_MAX_FEATURES
if 'use_char_features' in sig.parameters:
    build_kwargs['use_char_features'] = USE_CHAR_FEATURES
if 'n_jobs' in sig.parameters:
    build_kwargs['n_jobs'] = N_JOBS
else:
    print('Warning: old src/model.py detected in runtime; using legacy build_model signature.')

model = build_model(**build_kwargs)
print('Fitting model...')
model.fit(x_train, y_train)

print('Predicting validation/test...')
valid_pred = model.predict(x_valid)
test_pred = model.predict(x_test)
valid_metrics = evaluate_split('Validation', y_valid, valid_pred)
test_metrics = evaluate_split('Test', y_test, test_pred)

MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_OUT)

payload = {
    'model_type': 'lightgbm',
    'train_samples': len(x_train),
    'valid_samples': len(x_valid),
    'test_samples': len(x_test),
    'validation': valid_metrics,
    'test': test_metrics,
}
METRICS_OUT.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved model ->', MODEL_OUT)
print('Saved metrics ->', METRICS_OUT)


Model type: lightgbm (GBDT)
Train samples: 469,392
Valid samples: 58,674
Test samples: 58,675
Settings: svd=128, estimators=180, max_depth=6, word_max=120000, char_max=180000, n_jobs=2


TypeError: build_model() got an unexpected keyword argument 'word_max_features'